# Data preparation for the ensemble experiments

This notebook prepares the revised Arabic verb context–gloss dataset used in the individual-model and ensemble experiments. It validates the public schema, checks the labels and recreates the 80/10/10 split with seed 9.

The text is kept as a pair of sequences: `verb+context` and the candidate `sense`. Model-specific tokenization is performed in the next notebook.

## Setup

From the repository root, install the experiment dependencies with:

```bash
python -m pip install -e ".[experiments]"
```

In [ ]:
from pathlib import Path

import pandas as pd
from datasets import Dataset, DatasetDict

from arabic_wsd.data_quality import summarize_dataset
from arabic_wsd.experiments import load_pairs

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'arabic_verb_context_gloss_pairs_revised.xlsx'

pairs = load_pairs(DATA_PATH)
pairs.head()

## Quality checks

The loader removes the exported spreadsheet index, retains the three modeling columns, trims surrounding whitespace and validates binary labels. The summary below checks class counts, duplicates and context–gloss pairs associated with contradictory labels.

In [ ]:
summary = summarize_dataset(pairs)
pd.Series(summary.to_dict(), name='revised dataset').to_frame()

In [ ]:
label_distribution = (
    pairs['label']
    .value_counts()
    .sort_index()
    .rename_axis('label')
    .to_frame('instances')
)
label_distribution['share'] = label_distribution['instances'] / len(pairs)
label_distribution

## Recreate the experimental split

The revised dataset is divided into 80% training, 10% validation and 10% test data. The validation set is used to train the stacking model; the test set remains held out for final evaluation.

In [ ]:
dataset = Dataset.from_pandas(pairs, preserve_index=False)
train_and_temporary = dataset.train_test_split(test_size=0.20, seed=9)
validation_and_test = train_and_temporary['test'].train_test_split(test_size=0.50, seed=9)

splits = DatasetDict({
    'train': train_and_temporary['train'],
    'validation': validation_and_test['train'],
    'test': validation_and_test['test'],
})
splits

In [ ]:
split_summary = []
for name, split in splits.items():
    labels = pd.Series(split['label'])
    split_summary.append({
        'split': name,
        'instances': len(split),
        'positive': int((labels == 1).sum()),
        'negative': int((labels == 0).sum()),
        'positive_share': float(labels.mean()),
    })

pd.DataFrame(split_summary).set_index('split')

The split is recreated from the same dataset and seed in the following notebooks so that every base model and every ensemble is evaluated on the same examples.